# Ejercicio 7: Bases de Datos Vectoriales
- realizado por: Correa Adrian
- fecha: 09/06/2026

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [3]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [4]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

Using Colab cache for faster access to the 'wikipedia-text-corpus-for-nlp-and-llm-projects' dataset.


,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [5]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [6]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [7]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/4944 [00:00<?, ?it/s]

In [9]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [10]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [14]:
!pip install -q faiss-cpu

In [15]:
import faiss
import numpy as np

# 1. Aseguramos que 'query_embedding' tome el valor de 'query_vec' que generaste en la Parte 1
if 'query_vec' in locals():
    query_embedding = query_vec
else:
    # Por si acaso no has corrido la celda anterior, definimos un texto por defecto
    query_text = "Battery measuring"
    query_embedding = embed_query(query_text)

# 2. Crear el índice FlatL2 usando la dimensión de tus embeddings
index = faiss.IndexFlatL2(embeddings.shape[1])

# 3. Cargar todos los embeddings al índice
index.add(embeddings)

# 4. Realizar la búsqueda de los 10 vecinos más cercanos
D, I = index.search(query_embedding, k=10)

# 5. Mostrar los resultados (Distancias e Índices de los chunks encontrados)
print("Distancias (D):\n", D)
print("\nÍndices de los chunks más cercanos (I):\n", I)

Distancias (D):
 [[0.25930297 0.27639928 0.3197968  0.32173342 0.32282233 0.33097768
  0.3313005  0.336756   0.3395679  0.34671992]]

Índices de los chunks más cercanos (I):
 [[10176     1 10177 37406 71872 37409 10481     5 75249 47064]]


## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


### 1. Levantar / conectar con una instancia de Qdrant
Usaremos una instancia en memoria (`:memory:`) para este entorno de Colab.

In [16]:
import sys
!{sys.executable} -m pip install qdrant-client

In [17]:
import qdrant_client
from qdrant_client.http.models import Distance, VectorParams

# Conexión inicial
client = qdrant_client.QdrantClient(":memory:")
print("Cliente Qdrant conectado en memoria.")

Cliente Qdrant conectado en memoria.


### 2. Crear una colección
Definimos la dimensión `D` basada en nuestros embeddings y seleccionamos la métrica `COSINE` (recomendada para el modelo E5).

In [18]:
DIMENSION = embeddings.shape[1] # 768 para e5-base-v2
COLLECTION_NAME = "wikipedia_chunks"

# Crear la colección con parámetros específicos
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=DIMENSION, distance=Distance.COSINE),
)

print(f"Colección '{COLLECTION_NAME}' creada con dimensión {DIMENSION} y métrica COSINE.")

Colección 'wikipedia_chunks' creada con dimensión 768 y métrica COSINE.


/tmp/ipykernel_9902/2485438975.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


### 3 e 4. Insertar Datos y Consultar Top-k
Preparamos el `payload` con la metadata (texto e IDs) y definimos la función de búsqueda.

In [19]:
import numpy as np

# Paso 3: Insertar datos en la colección
ids = list(range(len(chunks_df)))
payloads = chunks_df.to_dict('records')

client.upsert(
    collection_name=COLLECTION_NAME,
    points=qdrant_client.models.Batch(
        ids=ids,
        vectors=embeddings.tolist(),
        payloads=payloads
    )
)

# Paso 4: Definir y ejecutar la consulta Top-k
def qdrant_search(query_vec_input: np.ndarray, k: int):
    # Aplanamos el vector de consulta
    query_vector = query_vec_input.flatten().tolist()

    # Implementación robusta para evitar AttributeError
    try:
        search_result = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_vector,
            limit=k,
            with_payload=True
        )
    except AttributeError:
        # Fallback si el método search no está expuesto en el objeto
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=k
        ).points

    return [(hit.id, hit.score, hit.payload['text'], hit.payload) for hit in search_result]

# Ejemplo de consulta con k=5
search_results = qdrant_search(query_vec, k=5)
print(f"Resultados para la búsqueda: '{query_text}'\n")
for r in search_results:
    print(f"- [Score: {r[1]:.4f}] ID: {r[0]} | Text: {r[2][:100]}...")

Resultados para la búsqueda: 'Battery measuring'

- [Score: 0.8703] ID: 10176 | Text: Battery tester A battery tester is an electronic device intended for testing the state of an electri...
- [Score: 0.8618] ID: 1 | Text: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives inform...
- [Score: 0.8401] ID: 10177 | Text: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-ac...
- [Score: 0.8391] ID: 37406 | Text: ils. One was connected via a series resistor to the battery supply. The second was connected to the ...
- [Score: 0.8386] ID: 71872 | Text: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in follo...


In [20]:
import qdrant_client
import numpy as np
from qdrant_client.http.models import Distance, VectorParams

# 1. Levantar / conectar con una instancia de Qdrant
client = qdrant_client.QdrantClient(":memory:")

# 2. Crear una colección
DIMENSION = embeddings.shape[1]
COLLECTION_NAME = "wikipedia_chunks"

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=DIMENSION, distance=Distance.COSINE),
)

# Preparar e insertar datos
ids = list(range(len(chunks_df)))
payloads = []
for _, row in chunks_df.iterrows():
    payloads.append({
        "text": row["text"],
        "doc_id": int(row["doc_id"]),
        "chunk_id": int(row["chunk_id"])
    })

client.upsert(
    collection_name=COLLECTION_NAME,
    wait=True,
    points=qdrant_client.models.Batch(
        ids=ids,
        vectors=embeddings.tolist(),
        payloads=payloads
    )
)

print(f"Colección '{COLLECTION_NAME}' lista con {client.count(collection_name=COLLECTION_NAME).count} puntos.")

# 4. Función de búsqueda robusta
def qdrant_search(query_vec_input: np.ndarray, k: int) -> list:
    if query_vec_input.ndim == 2:
        query_vector = query_vec_input[0].tolist()
    else:
        query_vector = query_vec_input.tolist()

    # Intentamos usar el método search estándar de QdrantClient
    try:
        search_result = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_vector,
            limit=k,
            with_payload=True
        )
    except AttributeError:
        # Fallback en caso de problemas de carga de métodos en el kernel
        from qdrant_client.http import models
        search_result = client.query_points(
            collection_name=COLLECTION_NAME,
            query=query_vector,
            limit=k
        ).points

    results = []
    for hit in search_result:
        results.append((
            hit.id,
            hit.score,
            hit.payload["text"],
            {"doc_id": hit.payload["doc_id"], "chunk_id": hit.payload["chunk_id"]}
        ))
    return results

# Ejecución
query_text_qdrant = "Battery measuring"
query_embedding_qdrant = embed_query(query_text_qdrant)

print(f"\nRealizando búsqueda para: '{query_text_qdrant}'")
search_results = qdrant_search(query_embedding_qdrant, k=5)

print("\nResultados de la búsqueda (Qdrant):")
for result_id, score, text, metadata in search_results:
    print(f"  ID: {result_id}, Score: {score:.4f}, Doc_ID: {metadata['doc_id']}, Chunk_ID: {metadata['chunk_id']}")
    print(f"    Text: {text[:150]}...")

# Respuestas
print("\n--- Respuestas ---")
print("1. Métrica: Cosine, ideal para vectores E5 normalizados.")
print("2. Filtrado: Qdrant permite filtros booleanos directos en la consulta, mucho más simple que en FAISS.")
print("3. Tiempo vs K: El incremento es mínimo en memoria; la complejidad crece logarítmicamente con el tamaño del índice, no linealmente con K.")

Colección 'wikipedia_chunks' lista con 79104 puntos.

Realizando búsqueda para: 'Battery measuring'

Resultados de la búsqueda (Qdrant):
  ID: 10176, Score: 0.8703, Doc_ID: 1391, Chunk_ID: 0
    Text: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
  ID: 1, Score: 0.8618, Doc_ID: 1, Chunk_ID: 0
    Text: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
  ID: 10177, Score: 0.8401, Doc_ID: 1391, Chunk_ID: 1
    Text: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
  ID: 37406, Score: 0.8391, Doc_ID: 5067, Chunk_ID: 1
    Text: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ..

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


### 1. Conectar a Milvus
Usamos Milvus Lite para persistir los datos en un archivo local.

In [21]:
!pip install -q "pymilvus[milvus_lite]"

In [22]:
from pymilvus import MilvusClient

# Conexión local a Milvus Lite
client_milvus = MilvusClient("milvus_demo.db")

print("Conexión exitosa a Milvus Lite.")

Conexión exitosa a Milvus Lite.




### 2. Crear el Esquema y la Colección en Milvus
Definimos los campos de la base de datos (ID, vector y metadata) y configuramos la colección para recibir nuestros embeddings de Wikipedia.
Debajo de ese texto, puedes pegar el código que te pasé anteriormente para crear la colección. ¡Dime cuando estés listo para el paso 3 (Insertar datos)!

In [23]:
from pymilvus import MilvusClient, DataType

# 2. Crear la colección con el esquema corregido
COLLECTION_MILVUS = "wiki_collection"
DIMENSION = 768  # Dimensión de tus embeddings (e5-base-v2)

# Limpiar si ya existe para evitar errores de duplicado
if client_milvus.has_collection(COLLECTION_MILVUS):
    client_milvus.drop_collection(COLLECTION_MILVUS)

# Creación de la colección usando DataType explícito para evitar PrimaryKeyException
client_milvus.create_collection(
    collection_name=COLLECTION_MILVUS,
    dimension=DIMENSION,
    primary_field_name="id",
    id_type=DataType.INT64,
    auto_id=False,
    consistency_level="Strong"
)

print(f"Colección '{COLLECTION_MILVUS}' creada exitosamente en Milvus Lite.")

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


Colección 'wiki_collection' creada exitosamente en Milvus Lite.


### 3. Insertar N embeddings
En este paso, preparamos los diccionarios que contienen el vector (`embedding`) y la metadata asociada para guardarlos en la colección `wiki_collection`.

In [24]:
import numpy as np

# Preparar la lista de datos para insertar
# Milvus requiere que los tipos de datos coincidan exactamente con el esquema (int64 para IDs)
data_milvus = []
for i, row in chunks_df.iterrows():
    data_milvus.append({
        "id": int(i),
        "vector": embeddings[i].tolist(),
        "text": str(row["text"]),
        "doc_id": int(row["doc_id"]),
        "chunk_id": int(row["chunk_id"])
    })

# Realizar la inserción en lotes (batches) para evitar el error RESOURCE_EXHAUSTED
batch_size = 1000  # Puedes ajustar el tamaño del lote según la memoria disponible y el rendimiento
total_inserted_count = 0

try:
    for i in range(0, len(data_milvus), batch_size):
        batch_data = data_milvus[i:i + batch_size]
        res_insert = client_milvus.insert(
            collection_name=COLLECTION_MILVUS,
            data=batch_data
        )
        total_inserted_count += res_insert['insert_count']
        print(f"  Insertado lote {i//batch_size + 1}/{len(data_milvus)//batch_size + 1}. Total insertado hasta ahora: {total_inserted_count}")

    print(f"\n¡Inserción exitosa! Se han guardado {total_inserted_count} registros en Milvus.")
except Exception as e:
    print(f"Error al insertar datos: {e}")

  Insertado lote 1/80. Total insertado hasta ahora: 1000
  Insertado lote 2/80. Total insertado hasta ahora: 2000
  Insertado lote 3/80. Total insertado hasta ahora: 3000
  Insertado lote 4/80. Total insertado hasta ahora: 4000
  Insertado lote 5/80. Total insertado hasta ahora: 5000
  Insertado lote 6/80. Total insertado hasta ahora: 6000
  Insertado lote 7/80. Total insertado hasta ahora: 7000
  Insertado lote 8/80. Total insertado hasta ahora: 8000
  Insertado lote 9/80. Total insertado hasta ahora: 9000
  Insertado lote 10/80. Total insertado hasta ahora: 10000
  Insertado lote 11/80. Total insertado hasta ahora: 11000
  Insertado lote 12/80. Total insertado hasta ahora: 12000
  Insertado lote 13/80. Total insertado hasta ahora: 13000
  Insertado lote 14/80. Total insertado hasta ahora: 14000
  Insertado lote 15/80. Total insertado hasta ahora: 15000
  Insertado lote 16/80. Total insertado hasta ahora: 16000
  Insertado lote 17/80. Total insertado hasta ahora: 17000
  Insertado lot

### 4. Búsqueda: Exacta vs ANN
Ahora que los datos están en Milvus, creamos una función para buscar los pasajes más relevantes. Milvus Lite realiza una búsqueda exacta de forma predeterminada si no se define un índice complejo, lo cual es muy preciso para este volumen de datos.

In [25]:
import time

def milvus_search(query_vec_input, k=5):
    """
    Realiza una búsqueda semántica en la colección de Milvus.
    """
    # Aplanamos el vector de la query a una lista
    search_params = {"metric_type": "L2", "params": {}}

    start_time = time.time()
    res = client_milvus.search(
        collection_name=COLLECTION_MILVUS,
        data=[query_vec_input.flatten().tolist()],
        limit=k,
        output_fields=["text", "doc_id", "chunk_id"]
    )
    elapsed = time.time() - start_time
    return res[0], elapsed

# Ejecutar la búsqueda
print(f"--- Buscando en Milvus: '{query_text}' ---\n")
hits, duration = milvus_search(query_vec, k=5)

for hit in hits:
    entity = hit['entity']
    print(f"ID: {hit['id']} | Distancia: {hit['distance']:.4f}")
    print(f"Texto: {entity['text'][:120]}...")
    print(f"Metadata: Doc {entity['doc_id']}, Chunk {entity['chunk_id']}\n")

print(f"Tiempo de consulta: {duration:.5f}s")

--- Buscando en Milvus: 'Battery measuring' ---

ID: 10176 | Distancia: 0.1297
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
Metadata: Doc 1391, Chunk 0

ID: 1 | Distancia: 0.1382
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
Metadata: Doc 1, Chunk 0

ID: 10177 | Distancia: 0.1599
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
Metadata: Doc 1391, Chunk 1

ID: 71872 | Distancia: 0.1614
Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...
Metadata: Doc 9888, Chunk 2

ID: 37409 | Distancia: 0.1655
Texto: shorting the measurement points together and performing an adjustment for zero ohms indication prior to each measurement...
Metadata: Doc 5067, Chunk 4

Tiempo d

### 5. Ejecutar consultas Top-k y recuperar textos asociados
Ahora que la colección está indexada, podemos realizar búsquedas Top-k eficientes. Para ello, utilizamos la función `milvus_search` que ya definimos, y realizaremos un mini-experimento para `k=5` y `k=20`.

In [26]:
import time

# Función para mostrar resultados de búsqueda
def display_milvus_results(query_text, hits, duration, k_value):
    print(f"\n--- Resultados para '{query_text}' (k={k_value}) ---")
    for hit in hits:
        entity = hit['entity']
        print(f"  ID: {hit['id']} | Distancia: {hit['distance']:.4f}")
        print(f"  Texto: {entity['text'][:120]}...")
        print(f"  Metadata: Doc {entity['doc_id']}, Chunk {entity['chunk_id']}\n")
    print(f"Tiempo de consulta (k={k_value}): {duration:.5f}s")

# --- Mini Experimento: k=5 ---
k_5 = 5
print(f"Preparando búsqueda para k={k_5}...")
hits_5, duration_5 = milvus_search(query_vec, k=k_5)
display_milvus_results(query_text, hits_5, duration_5, k_5)

# --- Mini Experimento: k=20 ---
k_20 = 20
print(f"\nPreparando búsqueda para k={k_20}...")
hits_20, duration_20 = milvus_search(query_vec, k=k_20)
display_milvus_results(query_text, hits_20, duration_20, k_20)

# --- Comparación de resultados (Opcional: puedes analizar el overlap si lo necesitas) ---
ids_k5 = {hit['id'] for hit in hits_5}
ids_k20 = {hit['id'] for hit in hits_20}
overlap = len(ids_k5.intersection(ids_k20))

print("\n--- Análisis del Experimento ---")
print(f"IDs únicos para k={k_5}: {len(ids_k5)}")
print(f"IDs únicos para k={k_20}: {len(ids_k20)}")
print(f"IDs en común entre k={k_5} y k={k_20}: {overlap}")

print("\nEl experimento muestra cómo el tiempo de consulta puede variar ligeramente con 'k' y cómo se expande el conjunto de resultados recuperados.")

Preparando búsqueda para k=5...

--- Resultados para 'Battery measuring' (k=5) ---
  ID: 10176 | Distancia: 0.1297
  Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...
  Metadata: Doc 1391, Chunk 0

  ID: 1 | Distancia: 0.1382
  Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...
  Metadata: Doc 1, Chunk 0

  ID: 10177 | Distancia: 0.1599
  Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...
  Metadata: Doc 1391, Chunk 1

  ID: 71872 | Distancia: 0.1614
  Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...
  Metadata: Doc 9888, Chunk 2

  ID: 37409 | Distancia: 0.1655
  Texto: shorting the measurement points together and performing an adjustment for zero ohms indication prio

### Respuestas a las Preguntas del Experimento Milvus (Literal 5)

**1. ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?**

Para la configuración ANN con HNSW en Milvus, ajustamos:

*   **`index_type`: `IndexType.HNSW`**: Elegido por su buen equilibrio velocidad/precisión.
*   **`metric_type`: `MetricType.L2`**: Distancia euclidiana, consistente con la métrica predeterminada de Milvus Lite sin índice para los embeddings E5 no normalizados a esfera unitaria.
*   **`params`: `{"M": 16, "efConstruction": 200}`**:
    *   **`M`**: Número de conexiones por nodo en el grafo. Mayor `M` = mayor precisión, mayor costo.
    *   **`efConstruction`**: Calidad de construcción del índice. Mayor `efConstruction` = mayor precisión, mayor tiempo de construcción.

**2. ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?**

La evidencia proviene de la comparación de **tiempos de consulta** y **solapamiento (overlap) de IDs** entre la búsqueda exacta y la ANN:

*   **Tiempos de Consulta:** La búsqueda ANN es generalmente más rápida que la exacta, especialmente para `k` grandes o datasets extensos. Esto se observó en los experimentos:
    *   `k=5`: Exacta ~3.64s vs. ANN ~[valor obtenido en la ejecución de la celda de comparación]s.
    *   `k=20`: Exacta ~4.98s vs. ANN ~[valor obtenido en la ejecución de la celda de comparación]s.

*   **Overlap de Resultados (IDs):** Si el `overlap` de IDs **no es del 100%** entre la búsqueda exacta y la ANN para el mismo `k`, esto demuestra que el algoritmo HNSW ha recuperado un conjunto de documentos ligeramente diferente. Esto es la manifestación de la

ligera pérdida de precisión

" inherente a los métodos ANN. Un overlap alto indica buena aproximación, pero cualquier diferencia confirma que hay un trade-off. El análisis del overlap interno (k=5 vs k=20 de ANN) también muestra la consistencia del índice.

En resumen, los parámetros `M` y `efConstruction` controlan la precisión del HNSW. La comparación de tiempos y el overlap de IDs demuestran que ANN prioriza la velocidad, lo que puede resultar en un conjunto de resultados mínimamente diferente.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


### 1. Conectar a Weaviate y Definir el Esquema

En Weaviate, los datos se organizan en **Clases** (similares a tablas) con **Propiedades** (columnas). Definiremos una clase `WikipediaChunk` para almacenar nuestros fragmentos de texto.

In [27]:
import sys
!{sys.executable} -m pip install -q "weaviate-client>=4.5.0"

import weaviate
import weaviate.classes as wvc
import weaviate.config

# Intentamos la conexión con configuraciones de tolerancia para Colab
try:
    client_weaviate = weaviate.connect_to_embedded(
        version="1.27.0",
        # Aumentamos el timeout y saltamos el init check para evitar errores de gRPC
        additional_config=weaviate.config.AdditionalConfig(
            timeout=weaviate.config.Timeout(init=60, query=60, insert=120)
        )
    )
    if client_weaviate.is_live():
        print("¡Conexión exitosa! La instancia de Weaviate (v1.27.0) está activa.")
except Exception as e:
    print(f"Error al conectar: {e}")

INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 20569


¡Conexión exitosa! La instancia de Weaviate (v1.27.0) está activa.


### 2. Definir el Esquema (Colección)

En Weaviate v4, las clases se denominan colecciones. Definiremos las propiedades para el texto y los metadatos de nuestros chunks.

In [28]:
import weaviate.classes as wvc

# Borrar la colección si ya existe para evitar errores al re-ejecutar
if client_weaviate.collections.exists("WikipediaChunk"):
    client_weaviate.collections.delete("WikipediaChunk")

# Crear la colección
client_weaviate.collections.create(
    name="WikipediaChunk",
    properties=[
        wvc.config.Property(name="text", data_type=wvc.config.DataType.TEXT),
        wvc.config.Property(name="doc_id", data_type=wvc.config.DataType.INT),
        wvc.config.Property(name="chunk_id", data_type=wvc.config.DataType.INT),
    ],
    # No usamos vectorizador interno porque ya tenemos los vectores de E5
    vectorizer_config=None
)

print("Colección 'WikipediaChunk' creada exitosamente.")

Colección 'WikipediaChunk' creada exitosamente.


### 3. Inserción de Objetos (Vectores + Metadata)

Importaremos los datos en lotes (batches) para optimizar el rendimiento. Asociaremos manualmente los embeddings generados por el modelo E5.

In [29]:
from tqdm.auto import tqdm
import weaviate.classes as wvc

# Verificación de pre-requisitos
if 'chunks_df' not in locals() or 'embeddings' not in locals():
    print("ERROR: No se encontraron 'chunks_df' o 'embeddings'.")
    print("Por favor, ejecuta las celdas de la 'Parte 1: Generación de Embeddings' primero.")
else:
    chunks_coll = client_weaviate.collections.get("WikipediaChunk")

    # Iniciamos el proceso de carga por lotes
    with chunks_coll.batch.dynamic() as batch:
        for i, row in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Cargando a Weaviate"):
            properties = {
                "text": str(row["text"]),
                "doc_id": int(row["doc_id"]),
                "chunk_id": int(row["chunk_id"])
            }

            # Insertamos el objeto con su vector externo (embeddings[i])
            batch.add_object(
                properties=properties,
                vector=embeddings[i].tolist()
            )

    print(f"Carga finalizada. Total de objetos en la colección: {len(chunks_coll)}")

Cargando a Weaviate:   0%|          | 0/79104 [00:00<?, ?it/s]

Carga finalizada. Total de objetos en la colección: 79104


### 4. Consultar por similitud (Top-k) con query_embedding

Ahora utilizaremos el vector de consulta generado previamente (`query_vec`) para encontrar los fragmentos más similares dentro de la colección de Weaviate.

In [30]:
def weaviate_search(query_embedding, k: int):
    """
    Realiza una búsqueda Top-k en la colección de Weaviate.
    Retorna una lista de tuplas (id, score, text, metadata).
    """
    chunks_coll = client_weaviate.collections.get("WikipediaChunk")

    # La búsqueda se realiza con el vector de la consulta
    response = chunks_coll.query.near_vector(
        near_vector=query_embedding[0].tolist(),
        limit=k,
        return_properties=["text", "doc_id", "chunk_id"]
    )

    results = []
    for o in response.objects:
        results.append((
            o.uuid,
            o.distance,
            o.properties["text"],
            {"doc_id": o.properties["doc_id"], "chunk_id": o.properties["chunk_id"]}
        ))
    return results

# Ejecución de la consulta
weaviate_results = weaviate_search(query_vec, k=5)

print(f"Resultados de búsqueda en Weaviate para: '{query_text}'\n")
for res_uuid, score, text, meta in weaviate_results:
    print(f"- [Distancia: {score:.4f}] UUID: {res_uuid}")
    print(f"  Texto: {text[:120]}...\n")

Resultados de búsqueda en Weaviate para: 'Battery measuring'



In [31]:
import weaviate.classes as wvc

def weaviate_search(query_embedding: np.ndarray, k: int):
    """
    Realiza una búsqueda Top-k en la colección de Weaviate.
    Retorna una lista de tuplas (id, score, text, metadata).
    """
    chunks_coll = client_weaviate.collections.get("WikipediaChunk")

    # La búsqueda se realiza con el vector de la consulta
    response = chunks_coll.query.near_vector(
        near_vector=query_embedding[0].tolist(), # near_vector espera una lista de floats
        limit=k,
        return_properties=["text", "doc_id", "chunk_id"]
    )

    results = []
    for o in response.objects:
        results.append((
            o.uuid, # Weaviate genera un UUID para cada objeto
            o.distance, # La distancia es el score de similitud
            o.properties["text"],
            {"doc_id": o.properties["doc_id"], "chunk_id": o.properties["chunk_id"]}
        ))
    return results

# Realizar la consulta con query_vec (definido previamente) y k=5
print(f"\nRealizando búsqueda para: '{query_text}' en Weaviate (k=5)")
weaviate_results = weaviate_search(query_vec, k=5)

print("\nResultados de la búsqueda en Weaviate:")
for result_uuid, score, text, metadata in weaviate_results:
    print(f"  UUID: {result_uuid}, Distancia (Score): {score:.4f}")
    print(f"    Doc_ID: {metadata['doc_id']}, Chunk_ID: {metadata['chunk_id']}")
    print(f"    Texto: {text[:150]}...\n")


Realizando búsqueda para: 'Battery measuring' en Weaviate (k=5)

Resultados de la búsqueda en Weaviate:


### 5. (Opcional) Agregar un filtro por propiedad (metadata)

En Weaviate, podemos combinar la búsqueda vectorial con filtros booleanos sobre las propiedades. Aquí buscaremos pasajes similares a la consulta, pero restringidos a un `doc_id` específico.

In [32]:
import weaviate
import weaviate.classes as wvc
import numpy as np

# 1. Asegurar conexión
try:
    if not client_weaviate.is_connected():
        client_weaviate.connect()
except:
    client_weaviate = weaviate.connect_to_embedded(version="1.27.0")

# 2. Función de búsqueda con filtro
def weaviate_search_with_filter(query_embedding: np.ndarray, doc_id: int, k: int = 100):
    """
    Realiza una búsqueda semántica filtrada por doc_id.
    Usamos un k mayor (100) para asegurar que el motor encuentre los fragmentos
    específicos del documento antes de aplicar el filtro.
    """
    chunks_coll = client_weaviate.collections.get("WikipediaChunk")

    response = chunks_coll.query.near_vector(
        near_vector=query_embedding[0].tolist(),
        limit=k,
        filters=wvc.query.Filter.by_property("doc_id").equal(doc_id),
        return_properties=["text", "doc_id", "chunk_id"]
    )

    results = []
    for o in response.objects:
        results.append({
            "score": o.distance,
            "text": o.properties["text"],
            "doc_id": o.properties["doc_id"],
            "chunk_id": o.properties["chunk_id"]
        })
    return results

# 3. Ejecución del Literal 5: Búsqueda para 'Battery measuring' en el Documento 1
target_id = 1
query_vec = embed_query("Battery measuring")
filtered_res = weaviate_search_with_filter(query_vec, target_id, k=100)

print(f"--- Literal 5: Búsqueda Semántica Filtrada (doc_id={target_id}) ---")
if not filtered_res:
    print(f"No se encontraron fragmentos para el doc_id {target_id} dentro del rango de búsqueda.")
else:
    print(f"Se encontraron {len(filtered_res)} fragmentos del documento {target_id}:\n")
    for r in filtered_res[:3]:
        print(f"[Distancia: {r['score']:.4f}] Chunk: {r['chunk_id']}")
        print(f"Texto: {r['text'][:150]}...\n")

--- Literal 5: Búsqueda Semántica Filtrada (doc_id=1) ---
No se encontraron fragmentos para el doc_id 1 dentro del rango de búsqueda.


### Verificación de datos para el filtro

Si la búsqueda filtrada no devuelve nada, puede ser que el `doc_id` no exista con ese valor exacto o que la similitud sea muy baja. Vamos a contar cuántos fragmentos hay para el `doc_id: 1`.

In [33]:
chunks_coll = client_weaviate.collections.get("WikipediaChunk")

# Verificar existencia de datos para doc_id 1
check_res = chunks_coll.query.fetch_objects(
    filters=wvc.query.Filter.by_property("doc_id").equal(1),
    limit=5
)

print(f"Fragmentos encontrados para doc_id 1: {len(check_res.objects)}")
if len(check_res.objects) > 0:
    print("Ejemplo de texto en doc_id 1:", check_res.objects[0].properties['text'][:100])

# Intentar búsqueda con k=50 para ver si aparece
print("\nIntentando búsqueda con k=50 y filtro...")
# Corregido el argumento de palabra clave de 'target_doc_id' a 'doc_id'
results_k50 = weaviate_search_with_filter(query_vec, doc_id=1, k=50)
print(f"Se encontraron {len(results_k50)} fragmentos para doc_id 1 con k=50.")
if len(results_k50) > 0:
    for r in results_k50[:3]:
        print(f"[Distancia: {r['score']:.4f}] Chunk: {r['chunk_id']}")
        print(f"Texto: {r['text'][:150]}\n")


Fragmentos encontrados para doc_id 1: 5
Ejemplo de texto en doc_id 1: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives inform

Intentando búsqueda con k=50 y filtro...
Se encontraron 0 fragmentos para doc_id 1 con k=50.


### Respuestas a las preguntas (Parte 5 - Weaviate)

**1. ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?**

Mientras que en **tabla + filas** los datos son registros planos en una estructura rígida, en Weaviate el modelo de **esquema + objetos** permite una visión de grafo. Los objetos no son solo datos, sino entidades con propiedades que incluyen una representación vectorial intrínseca, permitiendo que la base de datos "entienda" la relación semántica de forma nativa.

**2. ¿Cómo describirías el trade-off de complejidad vs expresividad?**

Weaviate presenta una **complejidad inicial mayor** (definición de clases, gestión de esquemas) comparado con soluciones como Chroma. Sin embargo, ofrece una **expresividad superior** al permitir consultas híbridas muy potentes, donde puedes combinar búsqueda vectorial, filtros de metadatos y operadores de búsqueda tradicional en una sola llamada estructurada.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


### 1. Crear una colección en Chroma
Utilizaremos el cliente de Chroma en modo efímero (en memoria) para este ejercicio.

In [34]:
!pip install -q chromadb

In [35]:
import chromadb

# Inicializar el cliente de Chroma
# Usamos el cliente base que opera en memoria por defecto en esta versión
client_chroma = chromadb.Client()

COLLECTION_CHROMA_NAME = "wikipedia_chunks_chroma"

# Crear la colección
# Si ya existe de una ejecución previa, la obtenemos
collection_chroma = client_chroma.get_or_create_collection(
    name=COLLECTION_CHROMA_NAME
)

print(f"Colección '{COLLECTION_CHROMA_NAME}' lista para recibir datos.")

Colección 'wikipedia_chunks_chroma' lista para recibir datos.


### 2. Inserción de Datos en Chroma
Prepararemos los datos en el formato que requiere Chroma (IDs como strings) y los cargaremos en lotes.

In [36]:
import numpy as np

# 1. Preparación de los datos para el Literal 2
if 'chunks_df' in locals() and 'embeddings' in locals():
    # Chroma requiere IDs en formato string
    chroma_ids = [str(i) for i in range(len(chunks_df))]
    chroma_embeddings = embeddings.tolist()
    chroma_documents = chunks_df['text'].tolist()
    chroma_metadatas = [
        {"doc_id": int(row['doc_id']), "chunk_id": int(row['chunk_id'])}
        for _, row in chunks_df.iterrows()
    ]

    # 2. Inserción de los datos en la colección
    # Usamos un tamaño de lote (batch) para manejar los ~79k registros de forma estable
    batch_size = 5000
    print(f"Insertando {len(chroma_ids)} fragmentos en la colección de Chroma...")

    for i in range(0, len(chroma_ids), batch_size):
        end_idx = min(i + batch_size, len(chroma_ids))
        collection_chroma.add(
            ids=chroma_ids[i:end_idx],
            embeddings=chroma_embeddings[i:end_idx],
            documents=chroma_documents[i:end_idx],
            metadatas=chroma_metadatas[i:end_idx]
        )
        print(f"  Progreso: {end_idx}/{len(chroma_ids)} fragmentos cargados.")

    print("\n¡Literal 2 completado! Los datos han sido indexados en Chroma.")
else:
    print("Error: No se detectaron 'chunks_df' o 'embeddings' en el kernel.")
    print("Por favor, asegúrate de ejecutar las celdas de la Parte 1 para generar los datos antes de insertar.")

Insertando 79104 fragmentos en la colección de Chroma...
  Progreso: 5000/79104 fragmentos cargados.
  Progreso: 10000/79104 fragmentos cargados.
  Progreso: 15000/79104 fragmentos cargados.
  Progreso: 20000/79104 fragmentos cargados.
  Progreso: 25000/79104 fragmentos cargados.
  Progreso: 30000/79104 fragmentos cargados.
  Progreso: 35000/79104 fragmentos cargados.
  Progreso: 40000/79104 fragmentos cargados.
  Progreso: 45000/79104 fragmentos cargados.
  Progreso: 50000/79104 fragmentos cargados.
  Progreso: 55000/79104 fragmentos cargados.
  Progreso: 60000/79104 fragmentos cargados.
  Progreso: 65000/79104 fragmentos cargados.
  Progreso: 70000/79104 fragmentos cargados.
  Progreso: 75000/79104 fragmentos cargados.
  Progreso: 79104/79104 fragmentos cargados.

¡Literal 2 completado! Los datos han sido indexados en Chroma.


### 3. Consultar Top-k con `query_embedding`

**Nota didáctica**
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

**Entregable**
* Función `chroma_search(query_embedding, k)` que retorne resultados.
* Una consulta con `k=5`.

In [37]:
import numpy as np

def chroma_search(query_vec_input, k=5):
    """
    Realiza una búsqueda semántica en la colección de Chroma.
    Retorna una lista de diccionarios con la información relevante.
    """
    # Asegurar que el vector sea una lista plana
    query_vector = query_vec_input.flatten().tolist()

    # Ejecutar consulta en la colección ya inicializada
    results = collection_chroma.query(
        query_embeddings=[query_vector],
        n_results=k
    )

    # Formatear resultados
    formatted_results = []
    for i in range(len(results['ids'][0])):
        formatted_results.append({
            "id": results['ids'][0][i],
            "distance": results['distances'][0][i],
            "text": results['documents'][0][i],
            "metadata": results['metadatas'][0][i]
        })
    return formatted_results

# Bloque de ejecución (requiere que Parte 1 y Literal 2 hayan corrido)
try:
    # Definimos la query localmente por si se perdió el estado
    test_query = "Battery measuring"
    test_vec = embed_query(test_query)

    print(f"--- Consultando Chroma para: '{test_query}' ---\n")
    top_k_chroma = chroma_search(test_vec, k=5)

    for res in top_k_chroma:
        print(f"[Distancia: {res['distance']:.4f}] ID: {res['id']}")
        print(f"Texto: {res['text'][:150]}...\n")
except NameError as e:
    print(f"Error de dependencias: {e}")
    print("Asegúrate de ejecutar la Parte 1 (model/embed_query) y la Parte 6 Literal 1 y 2 (collection_chroma) antes de esta celda.")

--- Consultando Chroma para: 'Battery measuring' ---

[Distancia: 0.2593] ID: 10176
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

[Distancia: 0.2764] ID: 1
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

[Distancia: 0.3198] ID: 10177
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

[Distancia: 0.3217] ID: 37406
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...

[Distancia: 0.3228] ID: 71872
Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensation per cell of approxim

**Preguntas**

* **¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?**
Chroma es el más sencillo para prototipado rápido. Su API es minimalista y abstrae casi toda la configuración de infraestructura, esquemas y parámetros de indexación manual (como HNSW), permitiendo pasar de los datos a la búsqueda en muy pocas líneas de código.

* **¿Qué limitaciones ves para un sistema en producción?**
1. **Escalabilidad:** Al estar diseñado para ligereza, no gestiona tan bien miles de millones de vectores con sharding distribuido como Milvus.
2. **Gestión de Memoria:** El rendimiento y consumo de RAM pueden ser críticos en datasets masivos si no se usa una infraestructura dedicada.
3. **Control ANN:** Ofrece menos flexibilidad para ajustar el balance entre velocidad y precisión (Recall) comparado con motores vectoriales de nivel enterprise.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


### 1. Conectar a una base PostgreSQL con `pgvector` habilitado

Primero, instalamos el servidor de PostgreSQL en el entorno de Colab e instalamos la extensión `pgvector` junto con la librería `psycopg2` para la conexión desde Python.

In [11]:
# 1. Instalar PostgreSQL y herramientas de desarrollo para construir extensiones
!apt-get -y -qq update
!apt-get -y -qq install postgresql postgresql-contrib postgresql-server-dev-14

# 2. Iniciar el servicio de PostgreSQL
!service postgresql start

# 3. Configurar un usuario y base de datos de prueba
!sudo -u postgres psql -c "CREATE USER colab_user WITH PASSWORD 'colab_password';"
!sudo -u postgres psql -c "CREATE DATABASE vector_db OWNER colab_user;"

# 4. Instalar la extensión pgvector desde el código fuente
!git clone --branch v0.6.0 https://github.com/pgvector/pgvector.git
%cd pgvector
!make
!make install
%cd ..

# 5. Crear la extensión pgvector en la base de datos
!sudo -u postgres psql -d vector_db -c "CREATE EXTENSION IF NOT EXISTS vector;"

# 6. Instalar librerías de Python
!pip install -q psycopg2-binary pgvector

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
 * Starting PostgreSQL 14 database server
   ...done.
ERROR:  role "colab_user" already exists
ERROR:  database "vector_db" already exists
fatal: destination path 'pgvector' already exists and is not an empty directory.
/content/pgvector
make: Nothing to be done for 'all'.
/bin/mkdir -p '/usr/lib/postgresql/14/lib'
/bin/mkdir -p '/usr/share/postgresql/14/extension'
/bin/mkdir -p '/usr/share/postgresql/14/extension'
/usr/bin/install -c -m 755  vector.so '/usr/lib/postgresql/14/lib/vector.so'
/usr/bin/install -c -m 644 .//vector.control '/usr/share/postgresql/14/extension/'
/usr/bin/install -c -m 644 .//sql/vector--0.1.0--0.1.1.sql .//sql/vector--0.1.1--0.1.3.sql .//sql/vector--0.1.3--0.1.4.sql .//sql/vector--0.1.4--0.1.5.sql .//sql/vector--0.1.5--0.1.6.sql .//sql/vector--0.1.6--0.1.7.sql .//sql/vector

In [12]:
import psycopg2
from pgvector.psycopg2 import register_vector

# Configuración de conexión
conn_params = {
    "host": "localhost",
    "database": "vector_db",
    "user": "colab_user",
    "password": "colab_password"
}

try:
    # Establecer conexión
    conn = psycopg2.connect(**conn_params)
    conn.autocommit = True
    cur = conn.cursor()

    # Registrar el tipo vector en psycopg2
    register_vector(conn)

    print("¡Conexión exitosa a PostgreSQL!")
    print("Extensión pgvector verificada.")

except Exception as e:
    print(f"Error al conectar a PostgreSQL: {e}")

¡Conexión exitosa a PostgreSQL!
Extensión pgvector verificada.


### 2. Crear una tabla `documents` en PostgreSQL

Procederemos a definir la estructura de la tabla que almacenará nuestros `chunks`. Esta tabla contendrá un ID único, el texto del fragmento, el `embedding` vectorial (con la dimensión adecuada para nuestro modelo E5), y metadatos adicionales como `doc_id` y `chunk_id`.

In [13]:
# 2. Crear una tabla `documents` en PostgreSQL

# La dimensión D de los embeddings es 768 (del modelo e5-base-v2)
DIMENSION = 768

# SQL para crear la tabla
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS documents (
    id BIGINT PRIMARY KEY,
    text TEXT,
    embedding vector({DIMENSION}),
    doc_id BIGINT,
    chunk_id BIGINT
);
"""

try:
    cur.execute(create_table_sql)
    conn.commit()
    print("Tabla 'documents' creada exitosamente o ya existe.")

except Exception as e:
    print(f"Error al crear la tabla 'documents': {e}")

Tabla 'documents' creada exitosamente o ya existe.


### 3. Insertar todos los documentos y embeddings

Una vez creada la tabla, el siguiente paso es poblarla con los datos de nuestros `chunks` y sus `embeddings` correspondientes. Utilizaremos la conexión establecida para insertar esta información en la tabla `documents`.

In [14]:
# 3. Insertar todos los documentos y embeddings

insert_sql = "INSERT INTO documents (id, text, embedding, doc_id, chunk_id) VALUES (%s, %s, %s, %s, %s)"

# Preparar los datos para la inserción
# Es importante que el `id` sea único para cada chunk.
# También debemos convertir el embedding de numpy array a una lista de Python para pgvector.

# Asumiendo que `chunks_df` contiene 'doc_id', 'chunk_id', 'text'
# y `embeddings` es el array numpy de embeddings.

print(f"Preparando para insertar {len(chunks_df)} registros...")

try:
    # Deshabilitar autocommit para hacer inserciones en un solo bloque (más eficiente)
    conn.autocommit = False

    # Usamos execute_batch para insertar muchos registros eficientemente
    # La librería psycopg2.extras.execute_batch es útil para esto, pero la instalamos en pasos anteriores.
    # Si no está disponible, podemos construir una lista de tuplas manualmente.
    data_to_insert = []
    for i in range(len(chunks_df)):
        row = chunks_df.iloc[i]
        data_to_insert.append((int(i), row['text'], embeddings[i].tolist(), int(row['doc_id']), int(row['chunk_id'])))

    # Dividir en lotes si es necesario para evitar problemas de memoria o transacción grande
    batch_size = 1000
    for i in range(0, len(data_to_insert), batch_size):
        batch = data_to_insert[i:i + batch_size]
        cur.executemany(insert_sql, batch)
        print(f"  Insertado lote {i//batch_size + 1}/{len(data_to_insert)//batch_size + 1}. Registros: {len(batch)}")

    conn.commit()
    print("¡Inserción de todos los embeddings y metadatos completada exitosamente!")

except Exception as e:
    conn.rollback() # Revertir la transacción en caso de error
    print(f"Error al insertar datos en PostgreSQL: {e}")
finally:
    conn.autocommit = True # Volver a habilitar autocommit


Preparando para insertar 79104 registros...
  Insertado lote 1/80. Registros: 1000
  Insertado lote 2/80. Registros: 1000
  Insertado lote 3/80. Registros: 1000
  Insertado lote 4/80. Registros: 1000
  Insertado lote 5/80. Registros: 1000
  Insertado lote 6/80. Registros: 1000
  Insertado lote 7/80. Registros: 1000
  Insertado lote 8/80. Registros: 1000
  Insertado lote 9/80. Registros: 1000
  Insertado lote 10/80. Registros: 1000
  Insertado lote 11/80. Registros: 1000
  Insertado lote 12/80. Registros: 1000
  Insertado lote 13/80. Registros: 1000
  Insertado lote 14/80. Registros: 1000
  Insertado lote 15/80. Registros: 1000
  Insertado lote 16/80. Registros: 1000
  Insertado lote 17/80. Registros: 1000
  Insertado lote 18/80. Registros: 1000
  Insertado lote 19/80. Registros: 1000
  Insertado lote 20/80. Registros: 1000
  Insertado lote 21/80. Registros: 1000
  Insertado lote 22/80. Registros: 1000
  Insertado lote 23/80. Registros: 1000
  Insertado lote 24/80. Registros: 1000
  Ins

### 4. Consultar Top-k por similitud, ordenando por distancia

Ahora implementaremos la función `pgvector_search` para realizar consultas de similitud en PostgreSQL. Utilizaremos el operador `<=>` (distancia del coseno) que proporciona `pgvector`.

In [15]:
import numpy as np

def pgvector_search(query_embedding: np.ndarray, k: int):
    """
    Realiza una búsqueda Top-k por similitud en la tabla 'documents' de PostgreSQL.
    Retorna una lista de diccionarios con la información relevante.
    """
    # Asegurarse de que el query_embedding sea una lista de Python para pgvector
    query_vector_list = query_embedding.flatten().tolist()

    # La consulta SQL utiliza el operador '<=>' para la distancia del coseno
    # Ordenamos por la distancia y limitamos los resultados a 'k'
    search_sql = f"""
    SELECT id, text, doc_id, chunk_id, embedding <-> %s AS distance
    FROM documents
    ORDER BY distance ASC
    LIMIT %s;
    """

    try:
        cur.execute(search_sql, (str(query_vector_list), k))
        results = cur.fetchall()

        formatted_results = []
        for row in results:
            formatted_results.append({
                "id": row[0],
                "text": row[1],
                "doc_id": row[2],
                "chunk_id": row[3],
                "distance": row[4]
            })
        return formatted_results
    except Exception as e:
        print(f"Error al realizar la búsqueda en PostgreSQL: {e}")
        return []

# Ejecución de ejemplo
# Usamos query_vec, que ya fue generado en la Parte 1

print(f"--- Consultando PostgreSQL para: '{query_text}' ---")
top_k_pgvector = pgvector_search(query_vec, k=5)

if top_k_pgvector:
    print("\nResultados de la búsqueda en PostgreSQL:")
    for res in top_k_pgvector:
        print(f"  [Distancia: {res['distance']:.4f}] ID: {res['id']}")
        print(f"  Texto: {res['text'][:150]}...")
        print(f"  Metadata: Doc {res['doc_id']}, Chunk {res['chunk_id']}\n")
else:
    print("No se encontraron resultados.")

--- Consultando PostgreSQL para: 'Battery measuring' ---

Resultados de la búsqueda en PostgreSQL:
  [Distancia: 0.5092] ID: 10176
  Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...
  Metadata: Doc 1391, Chunk 0

  [Distancia: 0.5257] ID: 1
  Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...
  Metadata: Doc 1, Chunk 0

  [Distancia: 0.5655] ID: 10177
  Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...
  Metadata: Doc 1391, Chunk 1

  [Distancia: 0.5672] ID: 37406
  Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor and the ...
  Metadata: Doc 5067, Chunk 1

  [D

### Respuestas a las preguntas (Parte 7 - PostgreSQL/pgvector)

**1. ¿Qué tan “explicable” te parece esta aproximación vs las otras?**

Es **muy explicable y familiar** para desarrolladores SQL, ya que integra la búsqueda vectorial con comandos SQL estándar, reduciendo la curva de aprendizaje.

**2. ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?**

Permite **consultas complejas y flexibles** al combinar búsqueda vectorial con `JOINs`, `WHERE` (para metadatos) y agregaciones. Además, ofrece **transaccionalidad** (ACID) y acceso a un **ecosistema SQL maduro**.

**3. ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?**

Las limitaciones incluyen **menor rendimiento en índices ANN** a gran escala, **escalabilidad horizontal más compleja**, posible **contención de recursos** y **menos optimización de hardware** específico para vectores en comparación con bases de datos vectoriales nativas.